# qrdqn_py12 — 複数シード一括学習 (F-6)

余っている GPU を使い、**複数シードを並列学習**して seed アンサンブル用のモデル群を作る。
checkpoint 間のばらつき（120k当たり/140k外れ）対策の本命＝独立モデルの多数決。

**重要 — 並列の真の制約は GPU ではなく System RAM（リプレイバッファ）**
- `buffer_size` × 観測(130×29×4B) がそのまま RAM になる。20万だと約6GB/run。
- Colab の System RAM は約12.7GBなので、**フルバッファだと1〜2本が限界**。
- 並列したいなら `--buffer-size` を小さく（例 3万〜5万）＋ `--optimize-memory`。

**手順**: GPU確認 → clone → 依存 → 並列学習 → モデルを file.io でDL

## 0. GPU / RAM 確認

In [1]:
!nvidia-smi
import psutil
print('CPU コア:', psutil.cpu_count(), '| System RAM(GB):', round(psutil.virtual_memory().total/1e9, 1))


Sun Jun  7 14:16:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. コード取得（GitHub clone）
プライベートrepoなので token を使う。token は GitHub → Settings → Developer settings →
Fine-grained token で、このrepoの **Contents: Read** だけ付ければよい。

In [1]:
# === 方法A: GitHub clone ===
REPO_URL = "https://github.com/Waipy252/resnet-dqn.git"
BRANCH = "main"

%cd /content
![ -d resnet-dqn ] && rm -rf resnet-dqn
!git clone --branch $BRANCH --depth 1 $REPO_URL
%cd /content/resnet-dqn/qrdqn_py12
!pwd && ls


/content
Cloning into 'resnet-dqn'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 41 (delta 8), reused 29 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 9.22 MiB | 16.74 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/resnet-dqn/qrdqn_py12
/content/resnet-dqn/qrdqn_py12
algo.py		     nikkei_cp_1997-01-01_2024-01-01_120000_steps.zip
calc_performance.py  notebooks
config.py	     pyproject.toml
data.py		     README.md
docker-compose.yml   requirements.txt
Dockerfile	     run_simulation.py
docs		     server.py
_eval_ensemble.py    test.py
_eval_one.py	     train_multi.py
eval.py		     uv.lock
main.py		     visualize.py


## 2. 依存インストール（torchは触らない / 旧gymは入れない）

In [2]:
!pip install -q \
    "stable-baselines3==2.8.0" \
    "sb3-contrib==2.8.0" \
    "gymnasium" "shimmy" \
    "yfinance" "pandas-datareader"
print('done')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 33.0 MB/s eta 0:00:00
done


## 3. 複数シード並列学習
`train_multi.py` が学習データを一度だけDLしてキャッシュ → 各シードを最大 `--jobs` 並列で実行。
各シードのモデルは `models/seed{S}/` に保存される。

**目安**
- `--jobs` は CPU コア数程度（Colab無料は2）。RAMに余裕がなければ 1〜2。
- `--buffer-size` × `--jobs` × 約6GB/20万 が RAM 消費の目安。例: buffer 3万・jobs 2 なら約2GB。
- まず短い `--timesteps`（例 5000）で**動作確認**してから本番（15万など）。

In [5]:
!git pull


remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 5.34 KiB | 2.67 MiB/s, done.
From https://github.com/Waipy252/resnet-dqn
   082bc90..341df82  main       -> origin/main
Updating 082bc90..341df82
Fast-forward
 qrdqn_py12/algo.py                     |   4 +
 qrdqn_py12/notebooks/train_multi.ipynb | 266 +++++++++++++++++++++++++++++----
 2 files changed, 243 insertions(+), 27 deletions(-)


In [6]:
# --- 動作確認（短時間）---
!python train_multi.py --seeds 0 1 --jobs 2 --timesteps 5000 --save-freq 5000 \
    --buffer-size 30000 --optimize-memory
import glob
print('produced:', sorted(glob.glob('models/seed*/*.zip')))


[master] データキャッシュあり: _train_cache.pkl
[master] seeds=[0, 1] jobs=2 timesteps=5000
[master] 起動 seed=0 (pid=2576) ログ: train_seed0.log
[master] 起動 seed=1 (pid=2577) ログ: train_seed1.log
[master] 終了 seed=0: OK
[master] 終了 seed=1: OK
[master] 全シード完了。models/seed*/ を確認。
produced: ['models/seed0/nikkei_cp_1997-01-01_2024-01-01_seed0.zip', 'models/seed0/nikkei_cp_1997-01-01_2024-01-01_seed0_5000_steps.zip', 'models/seed1/nikkei_cp_1997-01-01_2024-01-01_seed1.zip', 'models/seed1/nikkei_cp_1997-01-01_2024-01-01_seed1_5000_steps.zip']


In [3]:
# --- 本番（例: 5シード × 15万ステップ、2並列）---
# RAM/時間と相談して seeds・jobs・timesteps を調整する
!python train_multi.py --seeds 0 1 2 3 4 --jobs 2 --timesteps 150000 --save-freq 10000 \
    --buffer-size 50000 --optimize-memory


[master] 学習データDL中（初回のみ）...
/content/resnet-dqn/qrdqn_py12/data.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  test_data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/content/resnet-dqn/qrdqn_py12/data.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  us_10y = yf.download("^TNX", start=start, end=end)[["Close"]]
[*********************100%***********************]  1 of 1 completed
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas_datareader/fred.py", line 27, in read
    return self._read()
           ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas_datareader/fred.py", line 62, in _read
    [fetch_data(url, n) for url, n in zip(urls, names)], axis=1, join="outer"
     ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas_datareader/fred.py", line 41, in fetch_data
   

In [ ]:
# 進捗は各シードのログで確認（別セルで実行）
!tail -n 5 train_seed*.log 2>/dev/null; nvidia-smi | head -n 15


## 4. モデルをまとめてダウンロード（Drive不要）
`models/` 以下を1つの tar にまとめ、file.io にアップロード。返る JSON の `"link"` をコピーして
Mac側で `curl -L -o models_multi.tar.gz "<link>" && tar xzf models_multi.tar.gz` する。
> file.io は1回DLで消える。複数回DLしたいときは 0x0.st（コメント参照）。

In [ ]:
!tar czf models_multi.tar.gz models/ && ls -lh models_multi.tar.gz
